In [3]:

"""
Generate summary and sensitivity-analysis sheets from
ParamFree_5FoldCV_seed62.xlsx.

Usage:
    python generate_summaries.py input.xlsx

Example:
    python generate_summaries.py ParamFree_5FoldCV_seed62.xlsx
"""

import argparse
import os
import pandas as pd

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter


# ============================================================
# CONFIGURATION
# ============================================================

RESULTS_XLSX_FILE = "./Results/ParamFree_5FoldCV_seed62.xlsx"
SHEET_RESULTS = "Results"
START_ROW = 4              # Excel row containing the header
SEED = 62
N_FOLDS = 5

OUTPUT_SHEETS = [
    "Methodology",
    "Best_By_EER",
    "Best_by_R1FAR1",
    "EER_Sensitivity Analysis",
    "R1FAR1_Sensitivity Analysis",
]


# ============================================================
# EXCEL FORMATTING
# ============================================================

def format_worksheet(ws):
    """Apply consistent formatting to a worksheet."""

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    header_fill = PatternFill(
        fill_type="solid",
        fgColor="1F4E78"
    )

    header_font = Font(
        bold=True,
        color="FFFFFF"
    )

    thin_border = Border(
        bottom=Side(
            style="thin",
            color="D9E2F3"
        )
    )

    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )

    for row in ws.iter_rows():
        for cell in row:
            cell.border = thin_border
            cell.alignment = Alignment(
                vertical="center",
                wrap_text=True
            )

    ws.row_dimensions[1].height = 35

    # Automatically adjust column widths.
    for column_cells in ws.columns:
        max_length = 0
        column_letter = get_column_letter(
            column_cells[0].column
        )

        for cell in column_cells:
            if cell.value is not None:
                max_length = max(
                    max_length,
                    len(str(cell.value))
                )

        ws.column_dimensions[column_letter].width = min(
            max(max_length + 2, 12),
            45
        )


def format_numeric_columns(ws):
    """Format percentage and standard deviation columns."""

    headers = {
        cell.value: cell.column
        for cell in ws[1]
    }

    percentage_columns = [
        "EER (%)",
        "EER_std",
        "Rank-1 @ FAR=1%",
        "R1FAR1_std",
    ]

    for column_name in percentage_columns:
        if column_name not in headers:
            continue

        col_idx = headers[column_name]

        for row in range(2, ws.max_row + 1):
            cell = ws.cell(row=row, column=col_idx)

            if isinstance(cell.value, (int, float)):
                cell.number_format = "0.0000"


def format_sensitivity_sheet(ws):
    """Format sensitivity-analysis sheets."""

    ws.freeze_panes = "B2"
    ws.auto_filter.ref = ws.dimensions

    header_fill = PatternFill(
        fill_type="solid",
        fgColor="1F4E78"
    )

    header_font = Font(
        bold=True,
        color="FFFFFF"
    )

    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )

    for row in ws.iter_rows():
        for cell in row:
            cell.alignment = Alignment(
                horizontal="center",
                vertical="center"
            )

    ws.row_dimensions[1].height = 35

    for column_cells in ws.columns:
        max_length = 0
        col_letter = get_column_letter(
            column_cells[0].column
        )

        for cell in column_cells:
            if cell.value is not None:
                max_length = max(
                    max_length,
                    len(str(cell.value))
                )

        ws.column_dimensions[col_letter].width = min(
            max(max_length + 2, 12),
            30
        )

    # Four decimal places for sensitivity values.
    for row in ws.iter_rows(
        min_row=2,
        min_col=2
    ):
        for cell in row:
            if isinstance(cell.value, (int, float)):
                cell.number_format = "0.0000"


# ============================================================
# LOAD RESULTS
# ============================================================

def load_results(filepath):
    """
    Read the Results sheet.

    Excel row 4 is the header.
    pandas header=3 because indexing starts at zero.
    """

    df = pd.read_excel(
        filepath,
        sheet_name=SHEET_RESULTS,
        header=START_ROW - 1
    )

    # Remove fully empty rows and columns.
    df = df.dropna(how="all")
    df = df.dropna(axis=1, how="all")

    # Normalize column names.
    df.columns = [
        str(column).strip()
        for column in df.columns
    ]

    required_columns = [
        "Fold",
        "System",
        "Rule",
        "EER (%)",
        "Rank-1 @ FAR=1%",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            "Missing required columns: "
            + ", ".join(missing_columns)
        )

    # Ensure numeric data types.
    df["Fold"] = pd.to_numeric(
        df["Fold"],
        errors="coerce"
    )

    df["EER (%)"] = pd.to_numeric(
        df["EER (%)"],
        errors="coerce"
    )

    df["Rank-1 @ FAR=1%"] = pd.to_numeric(
        df["Rank-1 @ FAR=1%"],
        errors="coerce"
    )

    # Remove invalid rows.
    df = df.dropna(
        subset=[
            "Fold",
            "System",
            "Rule",
            "EER (%)",
            "Rank-1 @ FAR=1%",
        ]
    )

    # Keep only the expected 5 folds.
    df = df[
        df["Fold"].isin(range(1, N_FOLDS + 1))
    ].copy()

    return df


# ============================================================
# AGGREGATION
# ============================================================

def aggregate_per_configuration(df):
    """
    Compute mean and standard deviation across folds.

    Grouping:
        System + Rule

    Since parameter-free rules have no p parameter,
    the configuration is uniquely identified by System + Rule.
    """

    aggregated = (
        df.groupby(
            ["System", "Rule"],
            as_index=False
        )
        .agg(
            EER_mean=("EER (%)", "mean"),
            EER_std=("EER (%)", "std"),
            R1FAR1_mean=("Rank-1 @ FAR=1%", "mean"),
            R1FAR1_std=("Rank-1 @ FAR=1%", "std"),
            N_Folds=("Fold", "nunique"),
        )
    )

    return aggregated


def select_best_by_eer(aggregated):
    """
    Select minimum mean EER per system.

    EER and R1FAR1 are reported at the same
    selected System + Rule configuration.
    """

    idx = (
        aggregated.groupby("System")["EER_mean"]
        .idxmin()
    )

    best = aggregated.loc[idx].copy()

    best = best.sort_values(
        by="System"
    )

    best = best.rename(
        columns={
            "EER_mean": "EER (%)",
            "R1FAR1_mean": "Rank-1 @ FAR=1%",
        }
    )

    columns = [
        "System",
        "Rule",
        "EER (%)",
        "EER_std",
        "Rank-1 @ FAR=1%",
        "R1FAR1_std",
    ]

    return best[columns].reset_index(drop=True)


def select_best_by_r1far1(aggregated):
    """
    Select maximum mean Rank-1 @ FAR=1% per system.

    EER and R1FAR1 are reported at the same
    selected System + Rule configuration.
    """

    idx = (
        aggregated.groupby("System")["R1FAR1_mean"]
        .idxmax()
    )

    best = aggregated.loc[idx].copy()

    best = best.sort_values(
        by="System"
    )

    best = best.rename(
        columns={
            "EER_mean": "EER (%)",
            "R1FAR1_mean": "Rank-1 @ FAR=1%",
        }
    )

    columns = [
        "System",
        "Rule",
        "EER (%)",
        "EER_std",
        "Rank-1 @ FAR=1%",
        "R1FAR1_std",
    ]

    return best[columns].reset_index(drop=True)


# ============================================================
# SENSITIVITY ANALYSIS
# ============================================================

def create_eer_sensitivity(aggregated):
    """
    EER sensitivity analysis.

    Rows    : Systems
    Columns : Rules
    Values  : Mean EER across 5 folds

    Lower values indicate better EER.
    """

    sensitivity = aggregated.pivot(
        index="System",
        columns="Rule",
        values="EER_mean"
    )

    sensitivity = sensitivity.sort_index()

    sensitivity.index.name = "System"

    return sensitivity.reset_index()


def create_r1far1_sensitivity(aggregated):
    """
    Rank-1 @ FAR=1% sensitivity analysis.

    Rows    : Systems
    Columns : Rules
    Values  : Mean Rank-1 @ FAR=1% across 5 folds

    Higher values indicate better performance.
    """

    sensitivity = aggregated.pivot(
        index="System",
        columns="Rule",
        values="R1FAR1_mean"
    )

    sensitivity = sensitivity.sort_index()

    sensitivity.index.name = "System"

    return sensitivity.reset_index()


# ============================================================
# METHODOLOGY SHEET
# ============================================================

def create_methodology():
    """
    Create a detailed methodology sheet.

    The methodology documents the evaluation protocol
    represented by the Results sheet.
    """

    methodology = [
        [
            "Methodology — Parameter-Free Fusion Rules"
        ],
        [],
        [
            "1. Experimental Objective"
        ],
        [
            "Evaluate parameter-free fusion rules for "
            "eight multimodal fusion systems (S1–S8) "
            "using EER and Rank-1 @ FAR=1%."
        ],
        [],
        [
            "2. Dataset and Evaluation Protocol"
        ],
        [
            "The experiments use the NIST-BSSR1 biometric "
            "score dataset and a 5-fold subject-disjoint "
            "cross-validation protocol."
        ],
        [
            "The random seed is fixed to 62 to ensure "
            "reproducibility of the partitioning."
        ],
        [
            "Each fold uses approximately 80% of the "
            "subjects for training and 20% for testing."
        ],
        [
            "The five folds are non-overlapping at the "
            "subject level."
        ],
        [],
        [
            "3. Training and Normalization"
        ],
        [
            "For each fold, normalization parameters are "
            "estimated exclusively from the training "
            "subset."
        ],
        [
            "Min-max normalization is applied to the "
            "individual matcher scores using the training "
            "minimum and maximum."
        ],
        [
            "The same normalization parameters are "
            "subsequently applied to the corresponding "
            "test subset."
        ],
        [
            "This procedure prevents information leakage "
            "from the test subset into training."
        ],
        [],
        [
            "4. Fusion Systems"
        ],
        [
            "The Results sheet contains eight fusion "
            "systems, identified as S1–S8."
        ],
        [
            "Each system is evaluated using the eight "
            "parameter-free fusion rules."
        ],
        [
            "Each rule represents a fixed mathematical "
            "fusion operator without a tunable p parameter."
        ],
        [],
        [
            "5. Evaluation Metrics"
        ],
        [
            "EER (%): Equal Error Rate, expressed as a "
            "percentage. Lower values indicate better "
            "verification performance."
        ],
        [
            "Rank-1 @ FAR=1%: Rank-1 identification rate "
            "measured at a false acceptance rate of 1%."
        ],
        [
            "The standard deviation is calculated across "
            "the five evaluation folds."
        ],
        [],
        [
            "6. Best Configuration Selection"
        ],
        [
            "Best_By_EER: For each system, the rule with "
            "the minimum mean EER across all five folds "
            "is selected."
        ],
        [
            "Best_by_R1FAR1: For each system, the rule with "
            "the maximum mean Rank-1 @ FAR=1% across all "
            "five folds is selected."
        ],
        [
            "Both EER and Rank-1 are reported at the same "
            "selected system-rule configuration."
        ],
        [
            "The reported standard deviations correspond "
            "to the selected configuration across the "
            "five folds."
        ],
        [],
        [
            "7. Sensitivity Analysis"
        ],
        [
            "EER_Sensitivity Analysis reports the mean EER "
            "for every system-rule pair."
        ],
        [
            "R1FAR1_Sensitivity Analysis reports the mean "
            "Rank-1 @ FAR=1% for every system-rule pair."
        ],
        [
            "The sensitivity tables enable comparison "
            "of rule behavior across the eight systems."
        ],
        [],
        [
            "8. Reproducibility"
        ],
        [
            f"Random seed: {SEED}"
        ],
        [
            f"Number of folds: {N_FOLDS}"
        ],
        [
            "Source worksheet: Results"
        ],
        [
            f"Source header row: Excel row {START_ROW}"
        ],
    ]

    return pd.DataFrame(methodology)


# ============================================================
# WRITE OUTPUT WORKSHEETS
# ============================================================

def write_sheets(filepath, sheets):
    """
    Replace existing output sheets and create new ones.

    The original Results sheet is preserved.
    """

    # Load workbook and remove existing output sheets.
    workbook = load_workbook(filepath)

    for sheet_name in OUTPUT_SHEETS:
        if sheet_name in workbook.sheetnames:
            del workbook[sheet_name]

    workbook.save(filepath)

    # Write new sheets.
    with pd.ExcelWriter(
        filepath,
        engine="openpyxl",
        mode="a",
        if_sheet_exists="replace"
    ) as writer:

        for sheet_name, dataframe in sheets.items():

            if sheet_name == "Methodology":
                # Methodology is a one-column document.
                dataframe.to_excel(
                    writer,
                    sheet_name=sheet_name,
                    index=False,
                    header=False
                )
            else:
                dataframe.to_excel(
                    writer,
                    sheet_name=sheet_name,
                    index=False
                )

    # Apply formatting after writing.
    workbook = load_workbook(filepath)

    for sheet_name in OUTPUT_SHEETS:
        ws = workbook[sheet_name]

        if sheet_name == "Methodology":
            ws.column_dimensions["A"].width = 120

            for row in ws.iter_rows():
                for cell in row:
                    cell.alignment = Alignment(
                        vertical="top",
                        wrap_text=True
                    )

            # Highlight section headings.
            for row in range(1, ws.max_row + 1):
                value = ws.cell(row=row, column=1).value

                if (
                    isinstance(value, str)
                    and value[:2].isdigit()
                ):
                    ws.cell(
                        row=row,
                        column=1
                    ).font = Font(bold=True)

        elif "Sensitivity Analysis" in sheet_name:
            format_sensitivity_sheet(ws)

        else:
            format_worksheet(ws)
            format_numeric_columns(ws)

    workbook.save(filepath)


# ============================================================
# MAIN PIPELINE
# ============================================================

if __name__ == "__main__":

    filepath = RESULTS_XLSX_FILE
    print(f"Reading file: {filepath}")

    # 1. Load Results.
    df = load_results(filepath)

    print(f"Loaded {len(df)} result rows.")

    # 2. Aggregate each System + Rule.
    aggregated = aggregate_per_configuration(df)

    # 3. Best configuration by minimum mean EER.
    best_eer = select_best_by_eer(aggregated)

    # 4. Best configuration by maximum mean Rank-1.
    best_r1far1 = select_best_by_r1far1(aggregated)

    # 5. Sensitivity tables.
    eer_sensitivity = create_eer_sensitivity(
        aggregated
    )

    r1far1_sensitivity = create_r1far1_sensitivity(
        aggregated
    )

    # 6. Methodology.
    methodology = create_methodology()

    sheets = {
        "Methodology": methodology,
        "Best_By_EER": best_eer,
        "Best_by_R1FAR1": best_r1far1,
        "EER_Sensitivity Analysis": eer_sensitivity,
        "R1FAR1_Sensitivity Analysis": r1far1_sensitivity,
    }

    # 7. Write or update worksheets.
    write_sheets(filepath, sheets)

    print("\nSuccessfully created/updated:")

    for sheet_name in OUTPUT_SHEETS:
        print(f"  - {sheet_name}")

    print(f"\nUpdated file: {filepath}")

Reading file: ./Results/ParamFree_5FoldCV_seed62.xlsx
Loaded 320 result rows.

Successfully created/updated:
  - Methodology
  - Best_By_EER
  - Best_by_R1FAR1
  - EER_Sensitivity Analysis
  - R1FAR1_Sensitivity Analysis

Updated file: ./Results/ParamFree_5FoldCV_seed62.xlsx
